# Aula 13C — Simulador Interativo de Model Routing, Orchestration e Utility

**Da comparação entre modelos à engenharia experimental de sistemas compostos de IA**

Nesta versão, a Aula 13C deixa de ser apenas demonstrativa e passa a funcionar como um **laboratório interativo no Kaggle**. Você poderá alterar prioridades de qualidade, custo e latência, controlar o `quality gate` e observar como essas decisões mudam a arquitetura recomendada.

> Pergunta central: **qual combinação de modelos, regras e mecanismos de avaliação entrega valor suficiente com custo, latência e risco aceitáveis?**


## 1. Do modelo isolado ao sistema composto

```text
entrada
  ↓
router
  ↓
tarefa simples ─────→ modelo econômico
baixa confiança ────→ modelo premium
caso crítico ───────→ modelo + crítico/revisão
  ↓
quality gate
  ↓
resposta
```

Estratégias que vamos comparar: **single-small**, **single-frontier**, **cascade** e **critique**.


## 2. Estudo de caso — Project HydraFusion

O HydraFusion é usado aqui como estudo de caso contemporâneo de **multi-model orchestration**. O foco pedagógico não é declarar um vencedor, mas observar como qualidade, custo e latência entram simultaneamente na decisão.

Consulte a **Biblioteca Viva de Referências** em `docs/references/references.yaml` e o estudo de caso em `docs/case-studies/github-hydrafusion-multi-model-orchestration.md`.

> Benchmarks são evidência contextual. Dataset, configuração, preços e critérios podem alterar as conclusões.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError('Este laboratório requer ipywidgets, normalmente disponível no Kaggle.') from exc

print('Ambiente interativo carregado com sucesso.')


## 3. Modelo didático do simulador

Usaremos quatro arquiteturas com valores **didáticos e normalizados**, não preços reais de fornecedores.

A função de utilidade é:

`U = wq·Q − wc·C − wl·L`

Os pesos `wq`, `wc` e `wl` representam prioridades. O simulador normaliza automaticamente os pesos para somarem 1.

No `cascade`, o `quality gate` controla indiretamente a taxa de escalonamento: gate mais exigente → mais casos enviados ao modelo premium → tendência de maior qualidade, custo e latência.


In [ ]:
BASE_SYSTEMS = pd.DataFrame({
    'system': ['single_small', 'single_frontier', 'cascade', 'critique'],
    'quality': [0.82, 0.95, 0.90, 0.96],
    'cost': [0.20, 1.00, 0.45, 1.55],
    'latency': [0.25, 0.80, 0.55, 1.20],
})

def minmax(series):
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - lo) / (hi - lo)

def cascade_from_gate(gate):
    # gate em [0,1]. Quanto maior, mais rigoroso.
    escalation_rate = np.clip(0.08 + 0.84 * gate, 0, 1)
    cheap_q, premium_q = 0.82, 0.95
    cheap_c, premium_c = 0.20, 1.00
    cheap_l, premium_l = 0.25, 0.80
    quality = (1-escalation_rate)*cheap_q + escalation_rate*premium_q
    cost = cheap_c + escalation_rate*premium_c
    latency = cheap_l + escalation_rate*premium_l
    return escalation_rate, quality, cost, latency

def evaluate_systems(wq, wc, wl, gate):
    weights = np.array([wq, wc, wl], dtype=float)
    if weights.sum() == 0:
        weights = np.array([1.0, 1.0, 1.0])
    weights = weights / weights.sum()
    wq, wc, wl = weights

    df = BASE_SYSTEMS.copy()
    escalation_rate, q, c, l = cascade_from_gate(gate)
    mask = df['system'].eq('cascade')
    df.loc[mask, ['quality', 'cost', 'latency']] = [q, c, l]

    df['quality_norm'] = minmax(df['quality'])
    df['cost_norm'] = minmax(df['cost'])
    df['latency_norm'] = minmax(df['latency'])
    df['utility'] = wq*df['quality_norm'] - wc*df['cost_norm'] - wl*df['latency_norm']
    df = df.sort_values('utility', ascending=False).reset_index(drop=True)
    return df, escalation_rate, (wq, wc, wl)


## 4. 🎛️ Simulador interativo

Mova os controles e observe quatro coisas ao mesmo tempo:

1. qual arquitetura fica em primeiro lugar;
2. como muda a taxa de escalonamento do `cascade`;
3. o trade-off entre custo e qualidade;
4. a interpretação textual produzida a partir do cenário.

Experimente, por exemplo, colocar **qualidade = 90**, depois **custo = 90**, e então elevar o **quality gate**.


In [ ]:
style = {'description_width': '150px'}
layout = widgets.Layout(width='95%')

w_quality = widgets.IntSlider(value=60, min=0, max=100, step=5, description='Peso qualidade', style=style, layout=layout, continuous_update=False)
w_cost = widgets.IntSlider(value=25, min=0, max=100, step=5, description='Peso custo', style=style, layout=layout, continuous_update=False)
w_latency = widgets.IntSlider(value=15, min=0, max=100, step=5, description='Peso latência', style=style, layout=layout, continuous_update=False)
quality_gate = widgets.FloatSlider(value=0.45, min=0.0, max=1.0, step=0.05, description='Quality gate', readout_format='.2f', style=style, layout=layout, continuous_update=False)

preset = widgets.ToggleButtons(
    options=[('Equilibrado','balanced'), ('Qualidade','quality'), ('Custo','cost'), ('Latência','latency')],
    value='balanced', description='Preset', style={'description_width': '80px'}
)
reset_button = widgets.Button(description='Restaurar cenário', button_style='')
out = widgets.Output()

def interpretation(df, escalation_rate, weights, gate):
    winner = df.iloc[0]
    second = df.iloc[1]
    wq, wc, wl = weights
    dominant = ['qualidade', 'custo', 'latência'][int(np.argmax(weights))]
    gap = winner['utility'] - second['utility']
    if gap < 0.05:
        confidence = 'A decisão está apertada: pequenas mudanças de prioridade podem trocar o vencedor.'
    else:
        confidence = 'Há uma vantagem mais clara do primeiro colocado neste cenário.'
    gate_text = ('rigoroso' if gate >= 0.70 else 'moderado' if gate >= 0.35 else 'permissivo')
    return (
        f'**Recomendação:** `{winner.system}` lidera o ranking. '
        f'A prioridade dominante é **{dominant}**. {confidence}  \n'
        f'**Cascade:** gate {gate_text} (`{gate:.2f}`), com escalonamento estimado de **{escalation_rate:.0%}**. '
        'Esse valor representa quantos casos seguiriam para a camada premium no modelo didático.'
    )

def render(*_):
    df, escalation_rate, weights = evaluate_systems(w_quality.value, w_cost.value, w_latency.value, quality_gate.value)
    with out:
        clear_output(wait=True)
        display(Markdown(interpretation(df, escalation_rate, weights, quality_gate.value)))
        show = df[['system','quality','cost','latency','utility']].copy()
        show.insert(0, 'rank', np.arange(1, len(show)+1))
        display(show.round(3))

        fig, ax = plt.subplots(figsize=(8, 5))
        for _, row in df.iterrows():
            ax.scatter(row['cost'], row['quality'], s=100)
            ax.annotate(row['system'], (row['cost'], row['quality']), xytext=(6,6), textcoords='offset points')
        ax.set_xlabel('Custo relativo')
        ax.set_ylabel('Qualidade')
        ax.set_title('Mapa custo × qualidade')
        ax.grid(True, alpha=0.25)
        plt.show()

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(df['system'], df['utility'])
        ax.axhline(0, linewidth=1)
        ax.set_ylabel('Utility')
        ax.set_title('Ranking pela função de utilidade')
        plt.xticks(rotation=20)
        plt.show()

def set_preset(change):
    if change.get('name') != 'value':
        return
    presets = {
        'balanced': (60,25,15),
        'quality': (85,10,5),
        'cost': (40,50,10),
        'latency': (40,10,50),
    }
    w_quality.value, w_cost.value, w_latency.value = presets[change['new']]
    render()

def reset(_):
    preset.value = 'balanced'
    quality_gate.value = 0.45
    w_quality.value, w_cost.value, w_latency.value = 60,25,15
    render()

for control in [w_quality, w_cost, w_latency, quality_gate]:
    control.observe(render, names='value')
preset.observe(set_preset, names='value')
reset_button.on_click(reset)

controls = widgets.VBox([preset, w_quality, w_cost, w_latency, quality_gate, reset_button])
display(controls, out)
render()


## 5. Laboratório de sensibilidade do `quality gate`

O próximo experimento varre automaticamente o gate de 0 a 1. O objetivo é visualizar que **mais rigor não é grátis**: ele tende a elevar escalonamento, custo e latência.


In [ ]:
gates = np.linspace(0, 1, 21)
rows = []
for gate in gates:
    escalation_rate, quality, cost, latency = cascade_from_gate(gate)
    rows.append({
        'gate': gate,
        'escalation_rate': escalation_rate,
        'quality': quality,
        'cost': cost,
        'latency': latency,
    })
sensitivity = pd.DataFrame(rows)
display(sensitivity.iloc[::4].round(3))

plt.figure(figsize=(8,4))
plt.plot(sensitivity['gate'], sensitivity['quality'], marker='o')
plt.xlabel('Quality gate')
plt.ylabel('Qualidade estimada')
plt.title('Gate × qualidade')
plt.grid(True, alpha=0.25)
plt.show()

plt.figure(figsize=(8,4))
plt.plot(sensitivity['gate'], sensitivity['cost'], marker='o')
plt.xlabel('Quality gate')
plt.ylabel('Custo relativo')
plt.title('Gate × custo')
plt.grid(True, alpha=0.25)
plt.show()


## 6. Desafio de engenharia

Agora use o simulador para construir três políticas e registre seus resultados:

| Cenário | Prioridade | O que observar |
|---|---|---|
| A | máxima qualidade | custo necessário para ganhar qualidade |
| B | orçamento restrito | quanto de qualidade você aceita perder |
| C | resposta rápida | efeito da latência sobre a arquitetura escolhida |

Depois responda: **existe uma arquitetura que vence em todos os cenários?**


In [ ]:
minhas_conclusoes = {
    'cenario_A': '',
    'cenario_B': '',
    'cenario_C': '',
    'arquitetura_universal_existe': '',
    'justificativa': '',
}
minhas_conclusoes


## 7. Extensão para um sistema TIL real

Uma arquitetura futura pode substituir os valores didáticos por medições reais:

```text
TF-IDF + Naive Bayes
        ↓ baixa confiança
Transformer
        ↓ caso crítico
LLM ou revisão humana
```

O simulador pode então receber métricas de experimentos reais do TIL: F1, custo por inferência, tempo p50/p95, taxa de abstenção, taxa de escalonamento e revisão humana.

### Síntese

```text
Qual modelo tem a maior métrica?
            ↓
Qual política de roteamento entrega valor suficiente
com qualidade, custo, latência e risco aceitáveis?
```
